# Switching acquisition functions within a run: how much is there to gain?

## Description

**Question.** An amortized acquisition *policy* would pick the acquisition function (arm) at each stage of a BO run from the current state. That only pays off if some state-dependent choice of arms beats the best single arm. **How much better than the tuned best static arm B\* can a non-anticipating switching rule get, on tasks from the PFN's own prior?** If even a rule with privileged knowledge of the task gains little, a learned policy can't gain more, and the idea is not worth amortizing.

**Why a new design.** The predecessor's kill test (pfn-acqpolicy, notebooks 02–05) measured a *hindsight oracle*: the best of 81 branch sequences per task. That number was mostly luck. A null tree whose branches all ran the same arm, with only the nuisance differing, already reached 81–95% of the oracle's gain. The upper bound that survived was ≲ 5% (§16 there). The design here removes that selection effect by construction:
- **Non-anticipating controller.** Every 2d steps the controller copies the current state into K × R lanes (K = 6 arms, R = 8 rollouts). Each lane runs its arm for one segment and then B\* until the horizon. The controller executes the arm with the best mean rollout value. The rollouts use **fresh nuisance**: candidate sets and fantasy draws independent of the evaluated trajectory's, so nothing about the realized future leaks into the choice. This is one-step policy improvement over B\* (Bertsekas' rollout algorithm). Its gain over B\* is a *lower* bound on what optimal switching could gain, and an unbiased estimate of what this rule gains.
- **Two value models.**
  - *privileged*: rollouts query the true f, and the value is the final regret. This is an upper reference: switching with task knowledge.
  - *belief*: rollouts observe y sampled sequentially from the PFN's posterior predictive, and the value is the best fantasized y. It needs no task knowledge. **This is what a policy amortized over the prior could learn**, since the PFN's predictive *is* the prior's posterior predictive.
- **Null control (equal arms).** The same privileged controller choosing among 6 copies of B\*. Each copy is restricted to its own random half of the candidates (`B*@k`), redrawn per step and replicate, so the arm identities carry no information. Its gain over the static `B*@0` must be 0. A CI excluding 0 means the design leaks, and the experiment is INVALID.

**Setup.**
- **Tasks.** BNN-prior functions at d ∈ {2, 4, 6}. The portfolio and B\* come from `static_tune` (tuning split 0). The pilot uses split 2 (48 tasks/d) and confirm uses split 1, so B\* is **re-measured on fresh tasks**.
- **Budget.** n_init = 2d, T = 10d, 5 decisions (segments of 2d steps). Context ≤ 72 < 100: in range for pfn_large.
- **Members.** `task.method ∈ {static, privileged, belief, equal_arms}` × d. The `static` member runs the 6 portfolio arms, `B*@0` and random search on the same tasks, all paired (common random numbers per task and step).
- **Numerics.** fp32 throughout. TF32 would flip ~7% of near-tie choices, adding chaos (not bias) to the paired differences (log §1).

**What to look for.**
- **Primary metric.** The paired gain over the static B\* on the same tasks: 1 − geometric-mean ratio of final simple regrets (log10(regret + 1e-4)), with a 95% bootstrap CI over tasks.
- **Secondary.**
  - regret curves;
  - the gains of every static arm over B\*, i.e. how flat the top is;
  - which arms the controller picks per decision and d (e.g. does it explore early and exploit late?);
  - agreement between the belief and privileged choices.
- **Power.** The MDE at 80% power from the per-task sd of the log ratio.

**Decision rules (fixed before any run; `analysis/bo.py::switching_verdict`).**
- *Pilot* (split 2, 48 tasks/d): GO to confirm if the null CIs include 0 at every d. The pilot's sd of the log ratio sets the confirm size: the n for an MDE of 5%, capped by the GPU window (≤ 400 tasks/d). The pilot's effect sizes are not used for any decision.
- *Confirm* (split 1):
  - **INVALID** if any null CI excludes 0;
  - **GO** for amortizing a switching policy if the *belief* controller gains ≥ 5% over B\* with its CI above 0 at ≥ 2 of the 3 d;
  - **NO-GO** if the *privileged* controller, the reference with task knowledge, gains < 5% at ≥ 2 of the 3 d;
  - **INCONCLUSIVE** otherwise.

**Limits.**
- In-distribution only: BNN-prior tasks, noise-free.
- One-step lookahead over a fixed base. A multi-step or learned policy could do better than this controller (this is a lower bound on switching's value), but it has nothing better to learn from than the belief controller's signal.
- K = 6 arms. The rollout estimate has Monte-Carlo noise (R = 8), which dilutes, but does not bias, the gain.
- One nuisance seed.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/switching/<sweep_id>"  # set after `just snapshot runs/switching/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.